# tests to delete unwanted modelIDs from the modelSTAC

In [1]:
import pystac_client
import pandas as pd
from eo_processing.utils.storage import WEED_storage

In [2]:
username = 'VAULT_TOKEN'  # or your terrascope user name to hand over credentials manually
# details to STAC
stac_url:str = 'https://catalogue.weed.apex.esa.int'
collection_id:str = 'model-STAC'

In [3]:
# get all itemIDs from modelSTAC
client = pystac_client.Client.open(stac_url)

search = client.search(
    collections=[collection_id],
   fields=["id", "properties.topology", "properties.training_year", "properties.model_version", "properties.name_spatial_region", "properties.name_spatial_zone", "properties.modelID"],
)

# run the search
results = []
for item in search.items_as_dicts():
    results.append([item['id'], item['properties']['modelID'], item['properties']['topology'], item['properties']['training_year'], item['properties']['model_version'], item['properties']['name_spatial_region'], item['properties']['name_spatial_zone']])

df_result = pd.DataFrame(results, columns=['item_id', 'modelID', 'typology', 'training_year', 'model_version', 'spatial_region', 'spatial_zone'])

In [4]:
# get the overview
df_result.head()

,item_id,modelID,typology,training_year,model_version,spatial_region,spatial_zone
0,IUCNGET_global_v317_2024_NEO,IUCNGET_global_v317_2024_NEO,IUCNGET,2024,3.17,global,Neotropical
1,IUCNGET_global_v317_2024_IND,IUCNGET_global_v317_2024_IND,IUCNGET,2024,3.17,global,IndoMalesian
2,IUCNGET_global_v317_2024_HOL,IUCNGET_global_v317_2024_HOL,IUCNGET,2024,3.17,global,Holartic
3,IUCNGET_global_v317_2024_AFR,IUCNGET_global_v317_2024_AFR,IUCNGET,2024,3.17,global,African
4,EUNIS2021plus_panEU_v311_2024_STE,EUNIS2021plus_panEU_v311_2024_STE,EUNIS2021plus,2024,3.11,panEU,Steppic


In [5]:
# selecte items to delete
df_del = df_result[~(df_result['typology'] == 'IUCNGET')]

ldelete = df_del[df_del['model_version'] < 3].item_id.tolist()

In [6]:
ldelete

['EUNIS2021plus_panEU_v201_2024_OneZone',
 'EUNIS2021plus_EU_v1_2024_STE',
 'EUNIS2021plus_EU_v1_2024_PAN',
 'EUNIS2021plus_EU_v1_2024_MED',
 'EUNIS2021plus_EU_v1_2024_CON',
 'EUNIS2021plus_EU_v1_2024_BOR',
 'EUNIS2021plus_EU_v1_2024_ATL',
 'EUNIS2021plus_EU_v1_2024_ALP']

In [8]:
# init the storage object
st = WEED_storage(username=username, s3_bucket='model', project='WEED', stac_env='prod')

In [8]:
# now we delete the items we do not want
for item in ldelete:
    print(f'delete item: {item}')
    st.delete_collection_item(collection_name=collection_id, item_id=item)

delete item: EUNIS2021plus_panEU_v201_2024_OneZone
Item 'EUNIS2021plus_panEU_v201_2024_OneZone' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_STE
Item 'EUNIS2021plus_EU_v1_2024_STE' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_PAN
Item 'EUNIS2021plus_EU_v1_2024_PAN' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_MED
Item 'EUNIS2021plus_EU_v1_2024_MED' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_CON
Item 'EUNIS2021plus_EU_v1_2024_CON' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_BOR
Item 'EUNIS2021plus_EU_v1_2024_BOR' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_ATL
Item 'EUNIS2021plus_EU_v1_2024_ATL' deleted successfully from collection 'model-STAC'
delete item: EUNIS2021plus_EU_v1_2024_ALP
Item 'EUNIS2021plus_EU_v1_2024_ALP' deleted 

In [6]:
# delete collections id
ldelcol = ['test-V311', 'test-V311-jobdb', 'IUCNGET-v3-MECE']

In [9]:
for item in ldelcol:
    print(f'delete collection: {item}')
    st.delete_collection(collection_name=item)

delete collection: test-V311
Collection test-V311 deleted successfully
delete collection: test-V311-jobdb
Collection test-V311-jobdb deleted successfully
delete collection: IUCNGET-v3-MECE
Collection IUCNGET-v3-MECE deleted successfully
